In [1]:
from pathlib import Path
import pandas as pd
import re

# ========= 改成你的路徑 =========
input_csv = Path(r"/Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Tables/demo_tables/ethnicity_demographic_shares.csv")
output_tex = Path(r"/Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Tables/demo_tables/ethnicity_demographic_shares.tex")
# ===============================

def latex_escape(text):
    if pd.isna(text):
        return ""
    text = str(text)
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    pattern = re.compile("|".join(re.escape(k) for k in replacements))
    return pattern.sub(lambda m: replacements[m.group(0)], text)

# 讀入 CSV
df = pd.read_csv(input_csv)

# 需要欄位
cols = [
    "ethn_group",
    "p_female",
    "p_age1629",
    "p_age3049",
    "p_age5069",
    "p_age70p",
    "n",
]
missing = [c for c in cols if c not in df.columns]
if missing:
    raise ValueError(f"CSV 缺少欄位: {missing}")

df = df[cols].copy()

# 數值欄位轉 numeric
num_cols = ["p_female", "p_age1629", "p_age3049", "p_age5069", "p_age70p", "n"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 指定排序
ethnicity_order = [
    "British/English/Scottish/Welsh/Northern Irish",
    "Indian",
    "Pakistani",
    "Bangladeshi",
    "African",
    "Caribbean",
    "Any other white background",
]
df["ethn_group"] = pd.Categorical(
    df["ethn_group"],
    categories=ethnicity_order,
    ordered=True
)
df = df.sort_values("ethn_group").reset_index(drop=True)

# LaTeX escape
df["ethn_group"] = df["ethn_group"].astype(str).map(latex_escape)

# 輸出 LaTeX
lines = []
lines.append(r"\begin{table}[htbp]")
lines.append(r"\centering")
lines.append(r"\caption{Demographic Characteristics by Ethnicity}")
lines.append(r"\label{tab:ethnicity_demographic}")
lines.append(r"\begin{threeparttable}")
lines.append(r"\footnotesize")
lines.append(r"\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}p{5.4cm}rrrrrr}")
lines.append(r"\toprule")
lines.append(r"Ethnicity & Female & 16--29 & 30--49 & 50--69 & 70+ & N \\")
lines.append(r"\midrule")

for _, row in df.iterrows():
    eth = row["ethn_group"]
    female = "" if pd.isna(row["p_female"]) else f'{row["p_female"]:.2f}'
    age1629 = "" if pd.isna(row["p_age1629"]) else f'{row["p_age1629"]:.2f}'
    age3049 = "" if pd.isna(row["p_age3049"]) else f'{row["p_age3049"]:.2f}'
    age5069 = "" if pd.isna(row["p_age5069"]) else f'{row["p_age5069"]:.2f}'
    age70p = "" if pd.isna(row["p_age70p"]) else f'{row["p_age70p"]:.2f}'
    n = "" if pd.isna(row["n"]) else f'{int(round(row["n"])):,}'
    lines.append(f"{eth} & {female} & {age1629} & {age3049} & {age5069} & {age70p} & {n} \\\\")

lines.append(r"\bottomrule")
lines.append(r"\end{tabular*}")
lines.append(r"\begin{tablenotes}[flushleft]")
lines.append(r"\footnotesize")
lines.append(
    r"\item Notes: This table reports weighted percentages of sex and age-group composition by ethnicity. Female indicates the share of women. The remaining columns report the distribution across age groups. Percentages are weighted using the CA Covid survey weights, and \(N\) denotes the unweighted sample size."
)
lines.append(r"\end{tablenotes}")
lines.append(r"\end{threeparttable}")
lines.append(r"\end{table}")

latex_table = "\n".join(lines)

# 寫出 .tex
output_tex.write_text(latex_table, encoding="utf-8")

print(f"LaTeX table saved to: {output_tex}")
print()
print(latex_table)

LaTeX table saved to: /Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Tables/demo_tables/ethnicity_demographic_shares.tex

\begin{table}[htbp]
\centering
\caption{Demographic Characteristics by Ethnicity}
\label{tab:ethnicity_demographic}
\begin{threeparttable}
\footnotesize
\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}p{5.4cm}rrrrrr}
\toprule
Ethnicity & Female & 16--29 & 30--49 & 50--69 & 70+ & N \\
\midrule
British/English/Scottish/Welsh/Northern Irish & 53.53 & 16.76 & 27.96 & 37.70 & 17.58 & 12,575 \\
Indian & 45.23 & 27.99 & 43.71 & 24.90 & 3.41 & 435 \\
Pakistani & 42.66 & 58.10 & 30.10 & 11.27 & 0.53 & 263 \\
Bangladeshi & 54.77 & 42.92 & 51.68 & 3.97 & 1.44 & 100 \\
African & 42.84 & 15.20 & 60.39 & 17.38 & 7.03 & 111 \\
Caribbean & 70.18 & 19.76 & 25.57 & 54.21 & 0.47 & 131 \\
Any other white background & 58.53 & 8.78 & 56.01 & 28.48 & 6.73 & 417 \\
\bottomrule
\end{tabular*}
\begin{tablenotes}[flushleft]
\footnotesize
\item Notes: This table reports we